# Swaption Backtest (Conceal Last 6 Rows)
This notebook hides the final 6 rows, trains on earlier history only, predicts those hidden rows recursively, and compares predictions vs truth.


In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 12)


In [2]:
# Load local training data (already in repo)
df = pd.read_parquet("../data/level1.parquet").copy()
df["Date"] = pd.to_datetime(df["Date"], format="mixed", dayfirst=True)
df = df.sort_values("Date").reset_index(drop=True)

feature_cols = [c for c in df.columns if c != "Date"]
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

after_na = df[feature_cols].isna().sum().sum()
if after_na:
    # Forward/backward fill handles any sparse missing entries for modeling.
    df[feature_cols] = df[feature_cols].ffill().bfill()

print("Rows:", len(df), "Features:", len(feature_cols))
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Remaining NA:", int(df[feature_cols].isna().sum().sum()))


Rows: 494 Features: 224
Date range: 2050-01-01 00:00:00 to 2051-12-23 00:00:00
Remaining NA: 0


In [3]:
# Convert series into supervised learning format using a rolling lookback window.
# X_t uses the previous `lookback` rows to predict row t.
lookback = 12

def make_supervised(dataframe, feature_names, lookback_steps=12):
    values = dataframe[feature_names].values.astype(np.float64)
    dates = dataframe["Date"].values
    X, y, y_dates = [], [], []
    for t in range(lookback_steps, len(dataframe)):
        X.append(values[t - lookback_steps:t].reshape(-1))
        y.append(values[t])
        y_dates.append(dates[t])
    return np.array(X), np.array(y), np.array(y_dates)

X_all, y_all, y_dates = make_supervised(df, feature_cols, lookback_steps=lookback)
print("Supervised X shape:", X_all.shape)
print("Supervised y shape:", y_all.shape)


Supervised X shape: (482, 2688)
Supervised y shape: (482, 224)


In [4]:
# Backtest setup: conceal the final 6 rows as pseudo-test
horizon = 6
hide_start = len(df) - horizon

if hide_start <= lookback:
    raise ValueError("Not enough rows for lookback + concealed horizon.")

train_df = df.iloc[:hide_start].copy()
hidden_df = df.iloc[hide_start:].copy()  # kept for evaluation only

# Build supervised training samples only from the visible training window
X_train, y_train, dates_train = make_supervised(train_df, feature_cols, lookback_steps=lookback)

print("Visible training rows:", len(train_df))
print("Hidden rows:", len(hidden_df))
print("Training samples:", len(X_train))
print("Train end date:", train_df["Date"].iloc[-1])
print("Hidden date range:", hidden_df["Date"].iloc[0], "to", hidden_df["Date"].iloc[-1])


Visible training rows: 488
Hidden rows: 6
Training samples: 476
Train end date: 2051-12-14 00:00:00
Hidden date range: 2051-12-15 00:00:00 to 2051-12-23 00:00:00


In [5]:
# Model: scale features + regularized linear regression for multi-output targets
model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=2.0, random_state=42))
])

model.fit(X_train, y_train)
print("Model trained on visible data only.")


Model trained on visible data only.


In [6]:
# Recursively predict the concealed final 6 rows
history = train_df[feature_cols].values.astype(np.float64).copy()
concealed_preds = []

for _ in range(horizon):
    x_next = history[-lookback:].reshape(1, -1)
    y_next = model.predict(x_next)[0]
    concealed_preds.append(y_next)
    history = np.vstack([history, y_next])

concealed_preds = np.array(concealed_preds)

y_true_hidden = hidden_df[feature_cols].values.astype(np.float64)
mae_hidden = mean_absolute_error(y_true_hidden, concealed_preds)
rmse_hidden = np.sqrt(mean_squared_error(y_true_hidden, concealed_preds))

print(f"Concealed-6 MAE : {mae_hidden:.8f}")
print(f"Concealed-6 RMSE: {rmse_hidden:.8f}")

# Row-wise error summary for the 6 concealed steps
row_mae = np.mean(np.abs(y_true_hidden - concealed_preds), axis=1)
comparison_summary = pd.DataFrame({
    "Date": hidden_df["Date"].values,
    "Row_MAE": row_mae,
})
comparison_summary


Concealed-6 MAE : 0.00572347
Concealed-6 RMSE: 0.00669424


,Date,Row_MAE
0,2051-12-15,0.003642
1,2051-12-17,0.005032
2,2051-12-18,0.005659
3,2051-12-20,0.006818
4,2051-12-21,0.006854
5,2051-12-23,0.006335


In [7]:
# Build full comparison tables and save
pred_hidden_df = pd.DataFrame(concealed_preds, columns=feature_cols)
pred_hidden_df.insert(0, "Date", hidden_df["Date"].values)

actual_hidden_df = hidden_df[["Date"] + feature_cols].reset_index(drop=True)

# Long-form comparison for easy inspection
comparison_long = (
    actual_hidden_df.set_index("Date").stack().rename("actual").to_frame()
    .join(pred_hidden_df.set_index("Date").stack().rename("pred"))
    .reset_index()
    .rename(columns={"level_1": "instrument"})
)
comparison_long["abs_err"] = (comparison_long["actual"] - comparison_long["pred"]).abs()

pred_hidden_df.to_parquet("../data/level1_last6_pred.parquet", index=False)
comparison_long.to_parquet("../data/level1_last6_comparison_long.parquet", index=False)
print("Saved:")
print("- ../data/level1_last6_pred.parquet")
print("- ../data/level1_last6_comparison_long.parquet")

comparison_long.head(10)


Saved:
- ../data/level1_last6_pred.parquet
- ../data/level1_last6_comparison_long.parquet


,Date,instrument,actual,pred,abs_err
0,2051-12-15,Tenor : 10; Maturity : 0.0833333333333333,0.031773,0.032615,0.000842
1,2051-12-15,Tenor : 10; Maturity : 0.25,0.057429,0.058912,0.001483
2,2051-12-15,Tenor : 10; Maturity : 0.5,0.080756,0.082843,0.002087
3,2051-12-15,Tenor : 10; Maturity : 0.75,0.100204,0.102671,0.002466
4,2051-12-15,Tenor : 10; Maturity : 1,0.118918,0.121625,0.002707
5,2051-12-15,Tenor : 10; Maturity : 1.5,0.139398,0.142561,0.003163
6,2051-12-15,Tenor : 10; Maturity : 10,0.251467,0.255138,0.003670
7,2051-12-15,Tenor : 10; Maturity : 15,0.274069,0.278232,0.004163
8,2051-12-15,Tenor : 10; Maturity : 2,0.164510,0.167898,0.003389
9,2051-12-15,Tenor : 10; Maturity : 20,0.294160,0.299423,0.005263
